In [6]:
# make_training_log_figures.py
# Place this script in the same directory as:
#   training_log_SmallUNet.csv
#   training_log_MediumUNet.csv
#   training_log_DepthModelWithCues.csv

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------

try:
    ROOT = Path(__file__).resolve().parent
except NameError:
    ROOT = Path.cwd()

OUT_DIR = ROOT / "figures_training_logs"
OUT_DIR.mkdir(exist_ok=True)

LOG_FILES = {
    "SmallUNet": ROOT / "training_log_SmallUNet.csv",
    "MediumUNet": ROOT / "training_log_MediumUNet.csv",
    "DepthModelWithCues": ROOT / "training_log_DepthModelWithCues.csv",
}

DISPLAY_NAMES = {
    "SmallUNet": "SmallUNet",
    "MediumUNet": "MediumUNet",
    "DepthModelWithCues": "DepthModelWithCues",
}

METRICS = ["loss", "rmse", "si_rmse", "abs_rel"]

plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "legend.fontsize": 9,
    "axes.grid": True,
    "grid.alpha": 0.3,
})


# ---------------------------------------------------------
# Helpers
# ---------------------------------------------------------

def save_figure(fig, name):
    png_path = OUT_DIR / f"{name}.png"
    pdf_path = OUT_DIR / f"{name}.pdf"
    fig.savefig(png_path, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    print(f"Saved: {png_path}")
    print(f"Saved: {pdf_path}")


def get_lr_column(df):
    if "lr_after_scheduler" in df.columns:
        return "lr_after_scheduler"
    if "lr" in df.columns:
        return "lr"
    return None


def best_val(df, metric):
    val = df[df["phase"] == "val"].copy()
    if metric not in val.columns:
        return np.nan, np.nan
    idx = val[metric].idxmin()
    return float(val.loc[idx, metric]), int(val.loc[idx, "epoch"])


def final_value(df, phase, metric):
    sub = df[df["phase"] == phase].copy()
    if metric not in sub.columns or sub.empty:
        return np.nan
    return float(sub.iloc[-1][metric])


def unique_epoch_rows(df):
    return df.drop_duplicates("epoch").copy()


# ---------------------------------------------------------
# Load logs
# ---------------------------------------------------------

logs = {}
for model, path in LOG_FILES.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing log file: {path}")
    logs[model] = pd.read_csv(path)

print("\nLoaded logs:")
for model, df in logs.items():
    print(f"  {model}: {len(df)} rows, {df['epoch'].nunique()} epochs")


# ---------------------------------------------------------
# Build summary table
# ---------------------------------------------------------

summary_rows = []

for model, df in logs.items():
    epoch_rows = unique_epoch_rows(df)

    row = {
        "Model": DISPLAY_NAMES[model],
        "Img. size": int(df["img_size"].iloc[0]),
        "Batch": int(df["batch_size"].iloc[0]),
        "Train samples": int(df["max_train_samples"].iloc[0]),
        "Val samples": int(df["max_val_samples"].iloc[0]),
        "Avg epoch sec.": epoch_rows["epoch_elapsed_sec"].mean(),
        "Total hours": epoch_rows["epoch_elapsed_sec"].sum() / 3600.0,
    }

    for metric in METRICS:
        best, epoch = best_val(df, metric)
        row[f"Best val {metric}"] = best
        row[f"Best val {metric} epoch"] = epoch

    row["Final train loss"] = final_value(df, "train", "loss")
    row["Final val loss"] = final_value(df, "val", "loss")
    row["Final loss gap"] = row["Final val loss"] - row["Final train loss"]

    lr_col = get_lr_column(df)
    if lr_col is not None:
        row["Initial LR"] = float(df[lr_col].iloc[0])
        row["Final LR"] = float(df[lr_col].iloc[-1])
    else:
        row["Initial LR"] = np.nan
        row["Final LR"] = np.nan

    summary_rows.append(row)

summary = pd.DataFrame(summary_rows)
summary_path = OUT_DIR / "summary_metrics.csv"
summary.to_csv(summary_path, index=False)
print(f"\nSaved summary table: {summary_path}")

print("\nSummary:")
print(summary.to_string(index=False))


# ---------------------------------------------------------
# Save LaTeX main results table
# ---------------------------------------------------------

table_cols = [
    "Model",
    "Best val loss",
    "Best val rmse",
    "Best val si_rmse",
    "Best val abs_rel",
    "Avg epoch sec.",
    "Total hours",
]

table = summary[table_cols].copy()

# Bold best value for each metric where lower is better.
lower_is_better_cols = [
    "Best val loss",
    "Best val rmse",
    "Best val si_rmse",
    "Best val abs_rel",
]

best_indices = {
    col: table[col].astype(float).idxmin()
    for col in lower_is_better_cols
}

def fmt_num(x, decimals=4):
    if pd.isna(x):
        return "--"
    return f"{x:.{decimals}f}"

latex_lines = []
latex_lines.append(r"\begin{table}[t]")
latex_lines.append(r"    \centering")
latex_lines.append(r"    \caption{Quantitative comparison of the three trained models. Lower is better for all accuracy metrics. Runtime is measured on the same hardware.}")
latex_lines.append(r"    \label{tab:main_results}")
latex_lines.append(r"    \begin{tabular}{lcccccc}")
latex_lines.append(r"        \hline")
latex_lines.append(r"        Model & Val. Loss & RMSE & SI-RMSE & AbsRel & Sec./Epoch & Hours \\")
latex_lines.append(r"        \hline")

for idx, row in table.iterrows():
    cells = [row["Model"]]

    for col in lower_is_better_cols:
        value = fmt_num(row[col])
        if idx == best_indices[col]:
            value = r"\textbf{" + value + "}"
        cells.append(value)

    cells.append(fmt_num(row["Avg epoch sec."], decimals=1))
    cells.append(fmt_num(row["Total hours"], decimals=2))

    latex_lines.append("        " + " & ".join(cells) + r" \\")

latex_lines.append(r"        \hline")
latex_lines.append(r"    \end{tabular}")
latex_lines.append(r"\end{table}")

latex_table_path = OUT_DIR / "main_results_table.tex"
latex_table_path.write_text("\n".join(latex_lines))
print(f"Saved LaTeX table: {latex_table_path}")


# ---------------------------------------------------------
# Figure 1: Validation metric curves
# ---------------------------------------------------------

fig, axes = plt.subplots(2, 2, figsize=(11, 7), constrained_layout=True)
axes = axes.ravel()

metric_titles = {
    "loss": "Validation Loss",
    "rmse": "Validation RMSE",
    "si_rmse": "Validation SI-RMSE",
    "abs_rel": "Validation AbsRel",
}

for ax, metric in zip(axes, METRICS):
    for model, df in logs.items():
        val = df[df["phase"] == "val"]
        if metric not in val.columns:
            continue

        label = DISPLAY_NAMES[model]
        ax.plot(val["epoch"], val[metric], linewidth=2, label=label)

        best_value, best_epoch = best_val(df, metric)
        ax.scatter([best_epoch], [best_value], s=28)

    ax.set_title(metric_titles[metric])
    ax.set_xlabel("Epoch")
    ax.set_ylabel(metric)
    ax.legend()

save_figure(fig, "fig_validation_metric_curves")
plt.close(fig)


# ---------------------------------------------------------
# Figure 2: Train vs validation loss
# ---------------------------------------------------------

fig, ax = plt.subplots(figsize=(9, 5), constrained_layout=True)

for model, df in logs.items():
    train = df[df["phase"] == "train"]
    val = df[df["phase"] == "val"]

    ax.plot(
        train["epoch"],
        train["loss"],
        linestyle="--",
        linewidth=1.8,
        label=f"{DISPLAY_NAMES[model]} train",
    )
    ax.plot(
        val["epoch"],
        val["loss"],
        linestyle="-",
        linewidth=2.2,
        label=f"{DISPLAY_NAMES[model]} val",
    )

ax.set_title("Training and Validation Loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.legend(ncol=2)

save_figure(fig, "fig_train_val_loss")
plt.close(fig)


# ---------------------------------------------------------
# Figure 3: Final generalization gap
# ---------------------------------------------------------

models = list(logs.keys())
x = np.arange(len(models))
width = 0.35

final_train_loss = [final_value(logs[m], "train", "loss") for m in models]
final_val_loss = [final_value(logs[m], "val", "loss") for m in models]
gap = np.array(final_val_loss) - np.array(final_train_loss)

fig, ax = plt.subplots(figsize=(8, 5), constrained_layout=True)

ax.bar(x - width / 2, final_train_loss, width, label="Final train loss")
ax.bar(x + width / 2, final_val_loss, width, label="Final val loss")

for i, g in enumerate(gap):
    ax.text(
        x[i],
        max(final_train_loss[i], final_val_loss[i]) + 0.02,
        f"gap={g:.3f}",
        ha="center",
        va="bottom",
        fontsize=9,
    )

ax.set_title("Final Train--Validation Gap")
ax.set_ylabel("Loss")
ax.set_xticks(x)
ax.set_xticklabels([DISPLAY_NAMES[m] for m in models], rotation=15, ha="right")
ax.legend()

save_figure(fig, "fig_generalization_gap")
plt.close(fig)


# ---------------------------------------------------------
# Figure 4: Runtime vs accuracy tradeoff
# ---------------------------------------------------------

fig, ax = plt.subplots(figsize=(7, 5), constrained_layout=True)

for model, df in logs.items():
    epoch_rows = unique_epoch_rows(df)
    avg_epoch_sec = epoch_rows["epoch_elapsed_sec"].mean()
    best_loss, best_epoch = best_val(df, "loss")
    img_size = int(df["img_size"].iloc[0])

    ax.scatter(avg_epoch_sec, best_loss, s=80)
    ax.annotate(
        f"{DISPLAY_NAMES[model]}\n{img_size}x{img_size}",
        (avg_epoch_sec, best_loss),
        textcoords="offset points",
        xytext=(7, 7),
        fontsize=9,
    )

ax.set_title("Runtime--Accuracy Tradeoff")
ax.set_xlabel("Average epoch time, seconds")
ax.set_ylabel("Best validation loss")

save_figure(fig, "fig_runtime_accuracy_tradeoff")
plt.close(fig)


# ---------------------------------------------------------
# Figure 5: Learning-rate schedule
# ---------------------------------------------------------

fig, ax = plt.subplots(figsize=(9, 5), constrained_layout=True)

for model, df in logs.items():
    lr_col = get_lr_column(df)
    if lr_col is None:
        continue

    epoch_rows = unique_epoch_rows(df)
    ax.plot(
        epoch_rows["epoch"],
        epoch_rows[lr_col],
        linewidth=2,
        label=DISPLAY_NAMES[model],
    )

ax.set_title("Learning-Rate Schedule")
ax.set_xlabel("Epoch")
ax.set_ylabel("Learning rate")
ax.set_yscale("log")
ax.legend()

save_figure(fig, "fig_learning_rate_schedule")
plt.close(fig)


# ---------------------------------------------------------
# Figure 6: Component losses for models that logged them
# ---------------------------------------------------------

component_metrics = ["silog_loss", "l1_loss", "gradient_loss"]
component_models = [
    m for m, df in logs.items()
    if all(metric in df.columns for metric in component_metrics)
]

if component_models:
    x = np.arange(len(component_metrics))
    width = 0.8 / len(component_models)

    fig, ax = plt.subplots(figsize=(8, 5), constrained_layout=True)

    for i, model in enumerate(component_models):
        df = logs[model]
        vals = []
        for metric in component_metrics:
            best, _ = best_val(df, metric)
            vals.append(best)

        offset = (i - (len(component_models) - 1) / 2) * width
        ax.bar(x + offset, vals, width, label=DISPLAY_NAMES[model])

    ax.set_title("Best Validation Component Losses")
    ax.set_ylabel("Loss component value")
    ax.set_xticks(x)
    ax.set_xticklabels(["SiLog", "L1", "Gradient"])
    ax.legend()

    save_figure(fig, "fig_component_losses")
    plt.close(fig)

print("\nDone. Figures and tables are in:", OUT_DIR)


Loaded logs:
  SmallUNet: 200 rows, 100 epochs
  MediumUNet: 200 rows, 100 epochs
  DepthModelWithCues: 200 rows, 100 epochs

Saved summary table: C:\Users\ryanm\Downloads\CIL\figures_training_logs\summary_metrics.csv

Summary:
             Model  Img. size  Batch  Train samples  Val samples  Avg epoch sec.  Total hours  Best val loss  Best val loss epoch  Best val rmse  Best val rmse epoch  Best val si_rmse  Best val si_rmse epoch  Best val abs_rel  Best val abs_rel epoch  Final train loss  Final val loss  Final loss gap  Initial LR     Final LR
         SmallUNet        128      8           3000          500       50.530196     1.403617       0.908509                   90       0.127694                   69          0.915307                      81          4.003420                      52          0.621757        0.959547        0.337790       0.001 1.000000e-03
        MediumUNet        256      4          19600         3000      188.269388     5.229705       0.385157             